# Week 3: Deep Learning Architectures — LSTM vs BiLSTM
### Project: Intelligent Customer Review Sentiment Analysis
This notebook implements and evaluates:
1. **Model 1: Unidirectional LSTM** (Forward sequence learning)
2. **Model 2: Bidirectional LSTM with Attention Pooling** (Forward & backward context)
3. Training, validation loops, loss & accuracy curves
4. Comparative performance analysis

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using computational device: {device}')

## 1. PyTorch Dataset Definition

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

## 2. Model 1: Standard LSTM Architecture
$$h_t = \text{LSTM}(x_t, h_{t-1})$$
Captures left-to-right temporal dependencies in customer reviews.

In [ ]:
class PyTorchLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_classes=3, dropout=0.3):
        super(PyTorchLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, (hn, cn) = self.lstm(emb)
        # Use final hidden state
        feat = self.dropout(self.relu(self.fc1(hn[-1])))
        logits = self.out(feat)
        return logits

## 3. Model 2: Bidirectional LSTM (BiLSTM) with Attention Mechanism
$$\overrightarrow{h_t} = \text{LSTM}_{\text{fwd}}(x_t, \overrightarrow{h_{t-1}}), \quad \overleftarrow{h_t} = \text{LSTM}_{\text{bwd}}(x_t, \overleftarrow{h_{t+1}})$$
$$H_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$$

In [ ]:
class PyTorchBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_classes=3, dropout=0.35):
        super(PyTorchBiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.bilstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True, dropout=dropout)
        self.attention_weights = nn.Linear(hidden_dim * 2, 1)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        bilstm_out, _ = self.bilstm(emb)  # shape: (batch, seq_len, 2*hidden_dim)
        
        # Attention score computation
        attn_scores = self.attention_weights(bilstm_out)  # (batch, seq_len, 1)
        attn_probs = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_probs * bilstm_out, dim=1)  # (batch, 2*hidden_dim)
        
        feat = self.dropout(self.relu(self.fc1(context)))
        logits = self.out(feat)
        return logits

## 4. Training Loop & Validation Metrics

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seqs, labels in loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * seqs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

print('Training harness initialized successfully!')